In [1]:
import sys
import os

import tensorflow as tf

os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('variational_ae.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('encoder.py'), '..')))
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('decoder.py'), '..')))

import numpy as np
from sklearn.model_selection import train_test_split
import h5py

from variational_ae import VariationalAutoencoder
from encoder import EncoderBuilder
from decoder import DecoderBuilder

2025-08-16 14:45:37.950152: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-16 14:45:38.080679: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-16 14:45:40.560061: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"         # Keeps GPU order consistent
os.environ["CUDA_VISIBLE_DEVICES"] = "0"               # Makes only GPU 0 visible (useful even with 1 GPU)

gpus = tf.config.experimental.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # Prevents TF from using all GPU memory at once
    except RuntimeError as e:
        print("Error: ", e)
        exit(-1)

In [3]:
with h5py.File('Dataset/log_spec_data_dataset.h5', 'r') as h5f:
    log_spec_data_train = h5f['train'][:]
    log_spec_data_labels = h5f['label'][:]

In [4]:
log_spec_data_train = log_spec_data_train[..., np.newaxis]

log_spec_x_train, log_spec_x_val = train_test_split(log_spec_data_train, test_size=0.05, random_state=42)

In [5]:
print("Log Spec Train shape:", log_spec_x_train.shape)
print("Log Spec Validation shape:", log_spec_x_val.shape)

Log Spec Train shape: (28500, 256, 64, 1)
Log Spec Validation shape: (1500, 256, 64, 1)


In [6]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
EPOCHS = 50

In [7]:
input_shape = log_spec_x_train.shape[1:]
latent_space_dim = 128
decoder_out_filter = 1

In [8]:
# Hyperparameters for the Variational Autoencoder
recon_weight = 1.0  # Weight for the reconstruction loss.
beta = 1.0  # Weight for the KL divergence loss.

In [9]:
conv_layers_config=[
    {'filters': 512, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 256, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 128, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': 2},
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (2, 1)},
]

In [10]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))

I0000 00:00:1755326748.587653   25037 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1755326748.588632   25037 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [11]:
encoder = EncoderBuilder(latent_space_dim, conv_layers_config)
encoder(dummy_input)
shape_before_bottleneck = encoder.get_shape_before_bottleneck()
encoder_model = encoder.build_graph(input_shape=dummy_input.shape)

2025-08-16 14:45:49.873825: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90300


In [12]:
decoder = DecoderBuilder(shape_before_bottleneck, decoder_out_filter, conv_layers_config)
dummy_input = tf.random.normal((1, latent_space_dim))
decoder(dummy_input)
decoder_model = decoder.build_graph(input_shape=dummy_input.shape)

Number of Neurons:  <class 'numpy.int64'>
<class 'tuple'>
Value:  (8, 4, 32)
1st x's Shape:  TensorShape([1, 1024])
2nd x's Reshape:  TensorShape([1, 8, 4, 32])
1st x's Shape:  (None, 1024)
2nd x's Reshape:  (None, 8, 4, 32)


In [13]:
vae = VariationalAutoencoder(
        recon_weight=recon_weight,
        beta=beta,
        encoder=encoder_model,
        decoder=decoder_model
    )

In [14]:
dummy_input = tf.random.normal((1, input_shape[0], input_shape[1], input_shape[2]))
vae(dummy_input)

<tf.Tensor: shape=(1, 256, 64, 1), dtype=float32, numpy=
array([[[[0.4998241 ],
         [0.50002325],
         [0.50016344],
         ...,
         [0.5000109 ],
         [0.49974155],
         [0.49993545]],

        [[0.49999452],
         [0.5001384 ],
         [0.49995148],
         ...,
         [0.49980113],
         [0.49988   ],
         [0.49991477]],

        [[0.50012684],
         [0.50003254],
         [0.50023466],
         ...,
         [0.50042695],
         [0.49948162],
         [0.50018555]],

        ...,

        [[0.500169  ],
         [0.5001591 ],
         [0.50004786],
         ...,
         [0.4996384 ],
         [0.5007805 ],
         [0.49997562]],

        [[0.5003314 ],
         [0.50002587],
         [0.5008676 ],
         ...,
         [0.49968904],
         [0.49924958],
         [0.5003339 ]],

        [[0.5002438 ],
         [0.5000434 ],
         [0.49997061],
         ...,
         [0.50002396],
         [0.49998355],
         [0.5001222 ]]]], dtyp

In [15]:
from tensorflow.keras.optimizers import Adam
vae.compile(optimizer=Adam(learning_rate=0.0001))

In [16]:
history = vae.fit(
    x=log_spec_x_train,
    y=log_spec_x_train, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(log_spec_x_val, log_spec_x_val), # Validation data for monitoring
    shuffle=True
)

Epoch 1/50


2025-08-16 14:46:02.129032: I external/local_xla/xla/service/service.cc:163] XLA service 0x74cdcc00ad30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-16 14:46:02.129068: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-08-16 14:46:02.280000: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-16 14:46:03.148331: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-16 14:46:04.471469: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to loc

446/446 ━━━━━━━━━━━━━━━━━━━━ 154s 292ms/step - kl_loss: 62.1196 - reconstruction_loss: 718.1727 - total_loss: 780.2923 - val_val_kl_loss: 18.4551 - val_val_reconstruction_loss: 429.5935 - val_val_total_loss: 448.0486
Epoch 2/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 65s 147ms/step - kl_loss: 35.1298 - reconstruction_loss: 193.1423 - total_loss: 228.2721 - val_val_kl_loss: 33.6816 - val_val_reconstruction_loss: 176.0067 - val_val_total_loss: 209.6883
Epoch 3/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 62s 140ms/step - kl_loss: 32.6952 - reconstruction_loss: 173.5569 - total_loss: 206.2521 - val_val_kl_loss: 33.6463 - val_val_reconstruction_loss: 164.1049 - val_val_total_loss: 197.7512
Epoch 4/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 60s 134ms/step - kl_loss: 31.5984 - reconstruction_loss: 164.7753 - total_loss: 196.3737 - val_val_kl_loss: 32.9113 - val_val_reconstruction_loss: 156.2944 - val_val_total_loss: 189.2057
Epoch 5/50
446/446 ━━━━━━━━━━━━━━━━━━━━ 63s 140ms/step - kl_loss: 30.4374 - reconstruction_loss: 157.6

In [17]:
vae.save("initial_full_vae_log_spec.keras")
encoder, decoder = vae.get_models()
encoder.save("initial_encoder_vae_log_spec.keras")
decoder.save("initial_decoder_vae_log_spec.keras")